# 🚀 PRDetect — Huấn luyện & Đánh giá Pipeline GCN trên Kaggle

Notebook này triển khai đầy đủ luồng mô hình **GCN (Graph Convolutional Network - GCN2)** từ bài báo PRDetect (*Detecting AI-Generated Text via Perturbation and Syntactic Graph Neural Network*).

---

### 📌 Các tính năng chính trong Notebook:
1. **Thiết lập môi trường tự động**: Tự cài đặt `torch-geometric`, `transformers`, `spacy` và kiểm tra GPU.
2. **Cơ chế nạp dữ liệu thông minh (Smart Data Loader)**:
   - ⚡ **Nếu đã có file đồ thị (`.pkl`)**: Tự động phát hiện và nạp ngay trong 2 giây (từ `/kaggle/input/...` hoặc `./graph_data/`).
   - 🌐 **Nếu chưa có sẵn**: Tự động tải tập dữ liệu **HC3 (Hello-SimpleAI/HC3)** từ HuggingFace và trích xuất đồ thị cú pháp bằng **SpaCy + RoBERTa** chuẩn xác theo pipeline gốc `building_graph.py`.
3. **Mô hình GCN2 nguyên bản**: Giữ nguyên kiến trúc chuẩn của repo (`gcn.py`): 2 tầng `GCNConv` + Mean Pooling + Linear + Sigmoid.
4. **Quy trình Huấn luyện & Validation**: Tích hợp Early Stopping (patience=3), theo dõi Loss/Acc theo từng Epoch, tự động lưu Best Checkpoint.
5. **Đánh giá toàn diện trên Test Set**: Tính toán đầy đủ các chỉ số: **Accuracy, F1-Score, ROC-AUC, Precision, Recall, Confusion Matrix**.
6. **Trực quan hóa trực quan**: Biểu đồ Loss & Accuracy curves, Ma trận nhầm lẫn (Confusion Matrix Heatmap), Đường cong ROC-AUC.
7. **Demo suy luận thời gian thực (Inference Demo)**: Cho phép nhập bất kỳ đoạn văn bản tiếng Anh nào để kiểm tra xem là do **Người viết (Human)** hay do **AI sinh ra (AI/ChatGPT)**!

---
> ⚠️ **Lưu ý quan trọng trên Kaggle:**
> - Vào menu bên phải: **Notebook Settings** -> **Accelerator**: Chọn **GPU T4 x 2** hoặc **GPU P100**.
> - Bật **Internet**: **ON** để tải mô hình RoBERTa / SpaCy và dataset.


## 🛠️ Bước 1: Cài đặt thư viện cần thiết


In [ ]:
# Cài đặt PyTorch Geometric, HuggingFace transformers, datasets, và SpaCy
!pip install torch-geometric transformers datasets scikit-learn matplotlib seaborn -q
!python -m spacy download en_core_web_sm -q

print("✅ Đã cài đặt hoàn tất các thư viện!")


In [ ]:
import torch
import torch_geometric
import sklearn
import transformers
import spacy

print("=" * 50)
print(f"PyTorch version    : {torch.__version__}")
print(f"PyG version        : {torch_geometric.__version__}")
print(f"Transformers       : {transformers.__version__}")
print(f"CUDA khả dụng      : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Tên GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM khả dụng      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Đang chạy trên CPU! Vui lòng vào Kaggle Settings bật GPU để tăng tốc.")
print("=" * 50)


## 📂 Bước 2: Nạp dữ liệu đồ thị HC3 (Auto-detect hoặc Build trực tiếp)

Pipeline sẽ tự động quét:
1. Nếu bạn đã đính kèm Dataset Kaggle chứa `hc3_train.pkl`, `hc3_val.pkl`, `hc3_test.pkl` -> Sẽ nạp ngay lập tức.
2. Nếu chưa có -> Sẽ tự động tải từ HuggingFace `Hello-SimpleAI/HC3`, chuẩn bị 8,000 Train / 1,000 Val / 1,000 Test (Seed 2024 chuẩn theo `gcn.py`), sau đó trích xuất đồ thị cú pháp RoBERTa + SpaCy.


In [ ]:
import os
import glob
import pickle
import random
import time
import json as jsonlib
import numpy as np
import torch
from tqdm.notebook import tqdm

# Cấu hình số mẫu (nếu phải build từ đầu)
FAST_DEBUG_MODE = False  # Đổi thành True nếu bạn muốn chạy thử nghiệm nhanh với 500 train / 100 val / 100 test

N_TRAIN = 500 if FAST_DEBUG_MODE else 8000
N_VAL   = 100 if FAST_DEBUG_MODE else 1000
N_TEST  = 100 if FAST_DEBUG_MODE else 1000

# Tìm kiếm file .pkl có sẵn trong /kaggle/input hoặc thư mục hiện tại
possible_train_files = glob.glob("/kaggle/input/**/hc3_train.pkl", recursive=True) + glob.glob("./graph_data/hc3_train.pkl")
possible_val_files   = glob.glob("/kaggle/input/**/hc3_val.pkl", recursive=True) + glob.glob("./graph_data/hc3_val.pkl")
possible_test_files  = glob.glob("/kaggle/input/**/hc3_test.pkl", recursive=True) + glob.glob("./graph_data/hc3_test.pkl")

loaded_from_pkl = False

if possible_train_files and possible_val_files and possible_test_files:
    print(f"🔍 Đã tìm thấy dữ liệu đồ thị có sẵn:")
    print(f"  Train: {possible_train_files[0]}")
    print(f"  Val  : {possible_val_files[0]}")
    print(f"  Test : {possible_test_files[0]}")
    
    with open(possible_train_files[0], "rb") as f:
        hc3_train = pickle.load(f)
    with open(possible_val_files[0], "rb") as f:
        hc3_val = pickle.load(f)
    with open(possible_test_files[0], "rb") as f:
        hc3_test = pickle.load(f)
        
    loaded_from_pkl = True
    print("✅ Nạp dữ liệu đồ thị có sẵn thành công!")
else:
    print("ℹ️ Chưa tìm thấy file .pkl có sẵn. Bắt đầu tải và xây dựng đồ thị cú pháp từ HuggingFace HC3...")

if not loaded_from_pkl:
    # 1. Tải HC3 từ HuggingFace
    try:
        from datasets import load_dataset
        print("Đang tải HC3 JSONL...")
        hc3_dataset = load_dataset(
            'json',
            data_files='https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl',
            split='train'
        )
        raw_items = hc3_dataset
    except Exception as e:
        print(f"Lỗi tải trực tiếp: {e}. Thử wget...")
        import subprocess
        subprocess.run(['wget', '-q', '-O', '/tmp/hc3_all.jsonl',
                        'https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl'], check=True)
        raw_items = []
        with open('/tmp/hc3_all.jsonl', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    raw_items.append(jsonlib.loads(line.strip()))

    # 2. Chuyển đổi định dạng: Human = 1, AI = 0
    all_texts, all_labels = [], []
    for item in raw_items:
        for ans in item.get('human_answers', []):
            if ans and len(ans.strip()) > 30:
                all_texts.append(ans.strip())
                all_labels.append(1)  # Human = 1
        for ans in item.get('chatgpt_answers', []):
            if ans and len(ans.strip()) > 30:
                all_texts.append(ans.strip())
                all_labels.append(0)  # AI = 0

    combined = list(zip(all_texts, all_labels))
    random.seed(2024)
    random.shuffle(combined)

    assert len(combined) >= N_TRAIN + N_VAL + N_TEST, "Không đủ mẫu!"

    train_texts  = [t for t, _ in combined[:N_TRAIN]]
    train_labels = [l for _, l in combined[:N_TRAIN]]
    val_texts    = [t for t, _ in combined[N_TRAIN:N_TRAIN+N_VAL]]
    val_labels   = [l for _, l in combined[N_TRAIN:N_TRAIN+N_VAL]]
    test_texts   = [t for t, _ in combined[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST]]
    test_labels  = [l for _, l in combined[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST]]

    print(f"Tổng mẫu: Train={len(train_texts)}, Val={len(val_texts)}, Test={len(test_texts)}")

    # 3. Khởi tạo SpaCy và RoBERTa để trích xuất đồ thị
    import spacy
    from transformers import RobertaTokenizer, RobertaModel
    nlp = spacy.load("en_core_web_sm")
    device_build = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    roberta_model = RobertaModel.from_pretrained("roberta-base").to(device_build)
    roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base", do_lower_case=False)
    roberta_model.eval()

    def build_graph_data(texts, labels, desc="Building Graph"):
        all_token_embeddings = []
        all_edge_index = []
        valid_labels = []
        errors = 0

        for idx, text in enumerate(tqdm(texts, desc=desc)):
            try:
                doc = nlp(text)
                tokens = [token.text for token in doc]
                if len(tokens) < 3:
                    errors += 1
                    continue

                max_len = 512
                chunks = [tokens[i:i+max_len] for i in range(0, len(tokens), max_len)]
                chunk_embeds = []
                for chunk in chunks:
                    token_ids = roberta_tokenizer.convert_tokens_to_ids(chunk)
                    input_ids = torch.tensor(token_ids).unsqueeze(0).to(device_build)
                    with torch.no_grad():
                        out = roberta_model(input_ids)
                    chunk_embeds.append(out.last_hidden_state[0].cpu())
                token_embeddings = torch.cat(chunk_embeds, dim=0)

                edge0 = [word.i for word in doc]
                edge1 = [word.head.i for word in doc]
                edge_index = torch.tensor([edge0, edge1], dtype=torch.long)

                all_token_embeddings.append(token_embeddings)
                all_edge_index.append(edge_index)
                valid_labels.append(labels[idx])
            except Exception:
                errors += 1

        y = torch.tensor(valid_labels, dtype=torch.float32)
        print(f"  {desc} -> Hoàn thành: {len(valid_labels)} mẫu, Lỗi: {errors}")
        return {
            'all_token_embeddings': all_token_embeddings,
            'all_edge_index': all_edge_index,
            'y': y
        }

    t0 = time.time()
    print("⏳ Đang xây dựng đồ thị cú pháp cho Train, Val, Test...")
    hc3_train = build_graph_data(train_texts, train_labels, "HC3 Train")
    hc3_val   = build_graph_data(val_texts, val_labels, "HC3 Val")
    hc3_test  = build_graph_data(test_texts, test_labels, "HC3 Test")
    print(f"✅ Hoàn tất xây dựng đồ thị trong {(time.time() - t0)/60:.2f} phút!")

    # Lưu lại để tải lại nhanh trong các lần chạy sau
    os.makedirs("/kaggle/working/graph_data", exist_ok=True)
    with open("/kaggle/working/graph_data/hc3_train.pkl", "wb") as f:
        pickle.dump(hc3_train, f)
    with open("/kaggle/working/graph_data/hc3_val.pkl", "wb") as f:
        pickle.dump(hc3_val, f)
    with open("/kaggle/working/graph_data/hc3_test.pkl", "wb") as f:
        pickle.dump(hc3_test, f)
    print("💾 Đã lưu đồ thị vào /kaggle/working/graph_data/!")


In [ ]:
# Kiểm tra tổng quan về dữ liệu đồ thị
train_y = hc3_train['y']
val_y   = hc3_val['y']
test_y  = hc3_test['y']

print("📊 Thống kê tập dữ liệu đồ thị:")
print(f"  • Train : {len(train_y):,} mẫu (Human={int((train_y==1).sum())}, AI={int((train_y==0).sum())})")
print(f"  • Val   : {len(val_y):,} mẫu (Human={int((val_y==1).sum())}, AI={int((val_y==0).sum())})")
print(f"  • Test  : {len(test_y):,} mẫu (Human={int((test_y==1).sum())}, AI={int((test_y==0).sum())})")

sample_nodes = hc3_train['all_token_embeddings'][0].shape[0]
sample_feat  = hc3_train['all_token_embeddings'][0].shape[1]
sample_edges = hc3_train['all_edge_index'][0].shape[1]
print(f"\nVí dụ mẫu đồ thị đầu tiên: {sample_nodes} từ/nodes, {sample_edges} quan hệ cú pháp/edges, feature_dim={sample_feat}")


## 🧠 Bước 3: Định nghĩa Kiến trúc Mô hình GCN (GCN2 & GCN4)

Mô hình baseline chuẩn `GCN2` từ bài báo gốc `gcn.py`:
- **Đầu vào**: Đồ thị cú pháp câu, vector đặc trưng từ RoBERTa ($d = 768$).
- **Tầng GCN 1**: $768 \to 256$, kích hoạt ReLU + Dropout(0.5).
- **Tầng GCN 2**: $256 \to 64$, kích hoạt ReLU + Dropout(0.5).
- **Fully Connected (FC)**: $64 \to 1$.
- **Readout / Pooling**: Mean Pooling trên toàn bộ các node của câu: $h_{graph} = \frac{1}{|V|} \sum_{v \in V} h_v$.
- **Đầu ra**: Sigmoid hàm kích hoạt trả về xác suất $P(\text{Human})$. ($1 = \text{Human}, 0 = \text{AI}$).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

class GCN2(nn.Module):
    """
    Kiến trúc baseline GCN 2-layer nguyên bản trong PRDetect (gcn.py)
    """
    def __init__(self, input_dim=768, hidden_dim=256, output_dim=64, dropout_rate=0.5):
        super(GCN2, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.fc = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Tầng GCN 1
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Tầng GCN 2
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Phân loại và Mean Pooling toàn đồ thị
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        return torch.sigmoid(x)


class GCN4(nn.Module):
    """
    Kiến trúc GCN 4-layer mở rộng (tùy chọn)
    """
    def __init__(self, input_dim=768, hidden_dim=512, hidden_dim2=256, hidden_dim3=128, output_dim=64, dropout_rate=0.5):
        super(GCN4, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim2)
        self.conv3 = GCNConv(hidden_dim2, hidden_dim3)
        self.conv4 = GCNConv(hidden_dim3, output_dim)
        self.fc = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv4(x, edge_index))
        x = self.dropout(x)
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        return torch.sigmoid(x)

# Kiểm tra khởi tạo mô hình
test_sample_model = GCN2(768, 256, 64)
total_p = sum(p.numel() for p in test_sample_model.parameters())
print(f"✅ GCN2 khởi tạo thành công! Tổng số tham số (parameters): {total_p:,}")
del test_sample_model


## ⚙️ Bước 4: Thiết lập Huấn luyện & Siêu tham số (Hyperparameters)

Đồng bộ chính xác với `gcn.py`:
- `seed = 2024`
- `lr = 0.0001` (Adam Optimizer)
- `criterion = nn.BCELoss()`
- `epochs = 40`
- `patience = 3` (Early Stopping)


In [ ]:
import torch.optim as optim

seed = 2024
dataset_name = 'hc3'

# Đảm bảo tính tái lập (Reproducibility)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim   = 768
hidden_dim2 = 256
output_dim  = 64
learning_rate = 0.0001
epochs = 40
patience = 3

model = GCN2(input_dim=input_dim, hidden_dim=hidden_dim2, output_dim=output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCELoss()

model_dir = "/kaggle/working/model"
os.makedirs(model_dir, exist_ok=True)
best_model_path = os.path.join(model_dir, f"{dataset_name}_gcn_model_{seed}.pth")

print(f"🚀 Thiết bị: {device}")
print(f"📁 Đường dẫn lưu model: {best_model_path}")
print(f"🎯 Tham số huấn luyện: LR={learning_rate}, Epochs={epochs}, Early Stopping Patience={patience}")


## 🏋️ Bước 5: Vòng lặp Huấn luyện (Training Loop) & Validation


In [ ]:
train_len = len(hc3_train['y'])
val_len   = len(hc3_val['y'])

train_loss_history = []
val_loss_history   = []
train_acc_history  = []
val_acc_history    = []

val_max_acc = -1.0
tag = patience  # Early stopping counter

print(f"Bắt đầu huấn luyện: Train={train_len:,} mẫu | Val={val_len:,} mẫu")
print("=" * 70)

t_start = time.time()

for epoch in range(epochs):
    # ==================== TRAINING ====================
    model.train()
    epoch_loss = 0.0
    correct_predictions = 0

    train_pbar = tqdm(range(train_len), desc=f"Epoch {epoch+1:02d}/{epochs:02d} [Train]", leave=False)
    for i in train_pbar:
        data = Data(
            x=hc3_train['all_token_embeddings'][i],
            edge_index=hc3_train['all_edge_index'][i],
            y=hc3_train['y'][i]
        ).to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data.y.float().view(-1, 1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pred = (output >= 0.5).long()
        correct_predictions += (pred == data.y.view(-1, 1)).sum().item()

    epoch_loss /= train_len
    epoch_acc  = correct_predictions / train_len
    train_loss_history.append(epoch_loss)
    train_acc_history.append(epoch_acc)

    # ==================== VALIDATION ====================
    model.eval()
    val_loss = 0.0
    val_correct = 0

    val_pbar = tqdm(range(val_len), desc=f"Epoch {epoch+1:02d}/{epochs:02d} [Val]", leave=False)
    with torch.no_grad():
        for i in val_pbar:
            data = Data(
                x=hc3_val['all_token_embeddings'][i],
                edge_index=hc3_val['all_edge_index'][i],
                y=hc3_val['y'][i]
            ).to(device)

            output = model(data)
            loss = criterion(output, data.y.float().view(-1, 1))
            val_loss += loss.item()
            pred = (output >= 0.5).long()
            val_correct += (pred == data.y.view(-1, 1)).sum().item()

    val_loss /= val_len
    val_acc  = val_correct / val_len
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)

    # In kết quả epoch
    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # ==================== CHECKPOINT & EARLY STOPPING ====================
    if val_acc >= val_max_acc:
        val_max_acc = val_acc
        tag = patience  # Reset patience
        torch.save(model.state_dict(), best_model_path)
        print(f"  ⭐ Lưu Best Model Checkpoint tại Epoch {epoch+1}! (Val Acc kỷ lục: {val_max_acc*100:.2f}%)")
    else:
        tag -= 1
        print(f"  ⚠️ Val Acc không tăng ({val_acc*100:.2f}% < {val_max_acc*100:.2f}%). Patience còn: {tag}/{patience}")
        if tag == 0:
            print(f"\n⏹️ Early Stopping được kích hoạt tại Epoch {epoch+1}!")
            break

t_elapsed = time.time() - t_start
print("=" * 70)
print(f"🏁 Hoàn thành huấn luyện trong {t_elapsed/60:.2f} phút!")
print(f"🏆 Best Validation Accuracy: {val_max_acc*100:.2f}%")


## 🧪 Bước 6: Đánh giá chi tiết trên Test Set (Testing)

Nạp lại trọng số tốt nhất (`best_model_path`) và tính toán các chỉ số:
- **Test Loss**
- **Accuracy (Độ chính xác)**
- **F1-Score (F1 Nhị phân & F1 Macro)**
- **ROC-AUC (Diện tích dưới đường cong ROC)**
- **Bảng phân loại chi tiết (Classification Report)**


In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix, precision_score, recall_score
from datetime import datetime

test_len = len(hc3_test['y'])

# Khởi tạo lại mô hình test và nạp weights tốt nhất
test_model = GCN2(input_dim=input_dim, hidden_dim=hidden_dim2, output_dim=output_dim).to(device)
test_model.load_state_dict(torch.load(best_model_path, map_location=device))
test_model.eval()

test_loss = 0.0
correct_predictions = 0
test_preds_prob = []
test_preds_binary = []
test_labels = []

t_test_start = time.time()
with torch.no_grad():
    for i in tqdm(range(test_len), desc="Đang đánh giá Test Set"):
        data = Data(
            x=hc3_test['all_token_embeddings'][i],
            edge_index=hc3_test['all_edge_index'][i],
            y=hc3_test['y'][i]
        ).to(device)

        output = test_model(data)
        prob = output.item()
        test_preds_prob.append(prob)

        loss = criterion(output, data.y.float().view(-1, 1))
        test_loss += loss.item()

        pred_binary = 1 if prob >= 0.5 else 0
        test_preds_binary.append(pred_binary)
        target = int(data.y.item())
        test_labels.append(target)

        if pred_binary == target:
            correct_predictions += 1

t_test_time = time.time() - t_test_start

test_loss /= test_len
test_acc = correct_predictions / test_len
test_f1  = f1_score(test_labels, test_preds_binary)
test_auc = roc_auc_score(test_labels, test_preds_prob)
test_prec = precision_score(test_labels, test_preds_binary)
test_rec  = recall_score(test_labels, test_preds_binary)

print("=" * 60)
print("📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET (GCN BASELINE)")
print("=" * 60)
print(f"• Test Loss      : {test_loss:.4f}")
print(f"• Accuracy       : {test_acc*100:.2f}% ({correct_predictions}/{test_len})")
print(f"• ROC-AUC        : {test_auc:.4f}")
print(f"• F1-Score       : {test_f1:.4f}")
print(f"• Precision      : {test_prec:.4f}")
print(f"• Recall         : {test_rec:.4f}")
print(f"• Thời gian test : {t_test_time:.2f}s ({test_len/t_test_time:.1f} mẫu/s)")
print("=" * 60)

print("\n📄 BẢNG BÁO CÁO PHÂN LOẠI CHI TIẾT (Classification Report):")
print(classification_report(test_labels, test_preds_binary, target_names=["AI (0)", "Human (1)"], digits=4))

# Lưu vào file test_result.txt
with open("/kaggle/working/test_result.txt", "a", encoding="utf-8") as w:
    w.write(f"HC3_GCN\tAcc: {test_acc:.4f}\tAUC: {test_auc:.4f}\tF1: {test_f1:.4f}\tSeed: {seed}\t{datetime.now()}\n")
print("✅ Đã ghi nhận kết quả vào /kaggle/working/test_result.txt!")


## 📈 Bước 7: Trực quan hóa Kết quả (Visualization Plots)

Vẽ đồ thị:
1. Đường cong **Loss qua các Epoch** (Train vs Validation).
2. Đường cong **Accuracy qua các Epoch** (Train vs Validation).
3. **Confusion Matrix Heatmap** (Ma trận nhầm lẫn giữa AI và Human).
4. **Đường cong ROC (Receiver Operating Characteristic)** và giá trị AUC.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 1. Biểu đồ Loss
axes[0, 0].plot(train_loss_history, label='Train Loss', color='#1f77b4', linewidth=2.5)
axes[0, 0].plot(val_loss_history, label='Val Loss', color='#ff7f0e', linewidth=2.5, linestyle='--')
axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('BCE Loss', fontsize=12)
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# 2. Biểu đồ Accuracy
axes[0, 1].plot([a * 100 for a in train_acc_history], label='Train Acc', color='#2ca02c', linewidth=2.5)
axes[0, 1].plot([a * 100 for a in val_acc_history], label='Val Acc', color='#d62728', linewidth=2.5, linestyle='--')
axes[0, 1].axhline(y=val_max_acc * 100, color='gray', linestyle=':', label=f'Best Val Acc ({val_max_acc*100:.2f}%)')
axes[0, 1].set_title('Training & Validation Accuracy (%)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix Heatmap
cm = confusion_matrix(test_labels, test_preds_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1, 0],
            xticklabels=['Dự đoán AI (0)', 'Dự đoán Human (1)'],
            yticklabels=['Thực tế AI (0)', 'Thực tế Human (1)'],
            annot_kws={"size": 14, "weight": "bold"})
axes[1, 0].set_title(f'Confusion Matrix on Test Set (Total: {test_len})', fontsize=14, fontweight='bold')

# 4. ROC Curve
fpr, tpr, _ = roc_curve(test_labels, test_preds_prob)
axes[1, 1].plot(fpr, tpr, color='#9467bd', linewidth=2.5, label=f'GCN (AUC = {test_auc:.4f})')
axes[1, 1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.50)')
axes[1, 1].set_title('ROC Curve (Receiver Operating Characteristic)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
axes[1, 1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = "/kaggle/working/gcn_evaluation_results.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"📊 Đã lưu hình ảnh trực quan hóa độ nét cao vào: {plot_path}")


## 🔍 Bước 8: Thử nghiệm Suy luận Trực tiếp (Live Text Inference Demo)

Hàm `detect_text` cho phép bạn đưa vào bất kỳ câu hoặc đoạn văn bản tiếng Anh nào:
- Tự động tách token và phân tích cú pháp bằng SpaCy.
- Trích xuất embedding theo từng từ bằng RoBERTa.
- Đưa qua mô hình GCN đã huấn luyện.
- Trả về nhãn dự đoán (`Human-written` hay `AI-generated`) cùng xác suất phần trăm tin cậy.


In [ ]:
# Đảm bảo nlp và roberta đã sẵn sàng
import spacy
from transformers import RobertaTokenizer, RobertaModel

if 'nlp' not in globals():
    nlp = spacy.load("en_core_web_sm")
if 'roberta_model' not in globals():
    roberta_model = RobertaModel.from_pretrained("roberta-base").to(device)
    roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base", do_lower_case=False)
    roberta_model.eval()

def detect_text(raw_text, gcn_model, device_name=device):
    """
    Nhận raw_text -> Tạo đồ thị cú pháp -> Dự đoán Human vs AI
    """
    gcn_model.eval()
    doc = nlp(raw_text.strip())
    tokens = [token.text for token in doc]
    
    if len(tokens) < 3:
        return {"error": "Văn bản quá ngắn (cần ít nhất 3 từ) để tạo đồ thị cú pháp!"}

    # Trích xuất embeddings RoBERTa
    max_len = 512
    chunks = [tokens[i:i+max_len] for i in range(0, len(tokens), max_len)]
    chunk_embeds = []
    for chunk in chunks:
        token_ids = roberta_tokenizer.convert_tokens_to_ids(chunk)
        input_ids = torch.tensor(token_ids).unsqueeze(0).to(device_name)
        with torch.no_grad():
            out = roberta_model(input_ids)
        chunk_embeds.append(out.last_hidden_state[0].cpu())
    token_embeddings = torch.cat(chunk_embeds, dim=0)

    # Trích xuất cạnh quan hệ cú pháp SpaCy
    edge0 = [word.i for word in doc]
    edge1 = [word.head.i for word in doc]
    edge_index = torch.tensor([edge0, edge1], dtype=torch.long)

    # Đưa vào GCN
    graph_data = Data(x=token_embeddings, edge_index=edge_index).to(device_name)
    with torch.no_grad():
        human_prob = gcn_model(graph_data).item()
        
    ai_prob = 1.0 - human_prob
    pred_label = "Human (Người viết)" if human_prob >= 0.5 else "AI (Máy tạo - ChatGPT)"
    confidence = max(human_prob, ai_prob) * 100

    return {
        "text": raw_text,
        "prediction": pred_label,
        "human_probability": human_prob,
        "ai_probability": ai_prob,
        "confidence": confidence,
        "tokens_count": len(tokens)
    }

# ==================== THỬ NGHIỆM VỚI CÁC MẪU VĂN BẢN THỰC TẾ ====================
sample_texts = [
    # Mẫu 1: Do con người viết (tự nhiên, ngữ pháp tự do)
    "Honestly, I spent hours debugging this yesterday and found out that a single missing comma ruined the entire pipeline!",
    
    # Mẫu 2: Do AI/ChatGPT tạo (cấu trúc formal, cân đối)
    "Artificial intelligence is a branch of computer science that aims to create intelligent machines capable of performing tasks that typically require human intellect."
]

print("🔎 KIỂM TRA DỰ ĐOÁN TRỰC TIẾP:")
print("=" * 70)
for idx, text in enumerate(sample_texts, 1):
    res = detect_text(text, test_model)
    print(f"Mẫu #{idx}: "{text}"")
    print(f"  👉 Kết luận   : {res['prediction']} (Độ tin cậy: {res['confidence']:.2f}%)")
    print(f"  📊 Chi tiết   : P(Human) = {res['human_prob' if 'human_prob' in res else 'human_probability']*100:.2f}% | P(AI) = {res['ai_probability']*100:.2f}%")
    print("-" * 70)
